# Day 4: AI Agents - Manual OpenAI Implementation

Today we build an AI agent that can answer questions using our search tools from Day 3. We'll start with a **manual implementation** using the raw OpenAI API to understand the fundamentals before moving to frameworks.

## What We're Building

An FAQ agent that:
1. Receives user questions
2. Decides when to search (using text_search from Day 3)
3. Uses search results to generate answers
4. Handles errors gracefully

## Key Concepts

**Function Calling**: LLMs can invoke external functions (tools) to gather information. The LLM decides WHEN to call a tool based on the user's question.

**Stateless Pattern**: LLMs have no memory. Every API call must include the full conversation history (system prompt + user messages + tool calls + tool results).

**Tool Schema**: JSON description of a function (name, description, parameters) that the LLM uses to decide how to invoke it.

In [ ]:
import json
from openai import OpenAI
from typing import Any

# Reuse Day 3 data loading and search
from day1 import read_repo_data
from day2 import chunk_sliding_window
import minsearch

print("Imports successful")

In [ ]:
# Load DataTalksClub FAQ (same as Day 3)
print("Loading DataTalksClub FAQ...")
datatalk_docs = read_repo_data("DataTalksClub", "faq")
print(f"Loaded {len(datatalk_docs)} documents")

# Chunk documents
chunks = chunk_sliding_window(datatalk_docs, max_tokens=200, overlap=50)
print(f"Created {len(chunks)} chunks")

# Prepare for indexing (add title field)
def prepare_for_indexing(chunks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Add title field from metadata for field boosting."""
    for chunk in chunks:
        title = chunk.get("metadata", {}).get("title", "Untitled")
        chunk["title"] = title
    return chunks

prepared_chunks = prepare_for_indexing(chunks)

# Build text index
text_index = minsearch.Index(
    text_fields=["title", "content"],
    keyword_fields=["chunk_id"]
)
text_index.fit(prepared_chunks)
print(f"Text index built with {len(prepared_chunks)} chunks")

In [ ]:
def text_search(query: str, num_results: int = 5) -> list[dict[str, Any]]:
    """Search FAQ using text index from Day 3.

    Args:
        query: Search query string from user's question
        num_results: Maximum number of results to return (default: 5)

    Returns:
        List of dicts with title, content, score
    """
    # Use text_index built in previous cell
    results = text_index.search(
        query=query,
        boost_dict={"title": 2.0, "content": 1.0},
        num_results=num_results
    )
    # Return simplified results for agent
    return [
        {
            "title": r.get("title", "Untitled"),
            "content": r.get("content", "")[:500],  # Truncate for token efficiency
            "score": r.get("score", 0.0)
        }
        for r in results
    ]

# Test the function
test_results = text_search("Docker")
print(f"Test search for 'Docker' returned {len(test_results)} results")
print(f"Top result: {test_results[0]['title']}")

## Tool Schema

The LLM needs a JSON schema to understand what tools are available and how to call them. OpenAI's function calling uses this schema to:

1. **Decide** when to call a tool (based on user question)
2. **Generate** the correct arguments (query string)
3. **Validate** the arguments match the schema (strict mode)

In [ ]:
# Tool schema for text_search
# Per AGENT-01: JSON schema with name, description, parameters
tools = [
    {
        "type": "function",
        "function": {
            "name": "text_search",
            "description": "Search FAQ documentation using TF-IDF text search. Use this to find relevant information before answering questions about the FAQ.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query extracted from the user's question"
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return (default: 5)"
                    }
                },
                "required": ["query"],
                "additionalProperties": False  # Required for strict mode
            },
            "strict": True  # Guarantees schema adherence
        }
    }
]

print("Tool schema defined:")
print(json.dumps(tools, indent=2))

## System Prompt (Behavior Control)

The system prompt tells the agent HOW to behave. Key elements:
- **When** to use tools (always search before answering)
- **How** to use results (base answer on search results only)
- **Error handling** (what to do if search returns nothing)

In [ ]:
# System prompt controls agent behavior
# Per AGENT-03: Instructs agent to use search before answering
system_prompt = """You are a helpful FAQ assistant. Follow these rules strictly:

1. ALWAYS use the text_search tool before answering any question
2. Base your answer ONLY on the search results, never your training data
3. If search returns no relevant results, say "I don't have information about that in the FAQ"
4. Cite specific information from search results in your answer
5. If a tool returns an error, explain the issue and suggest the user rephrase

Never make up information. Never guess. Only answer based on search results."""

print("System prompt defined:")
print(system_prompt)

## The Agent Loop

This is the core pattern:

1. **User asks question** -> Add to messages
2. **Call LLM** -> Pass tools and messages
3. **LLM decides** -> Either call tool OR return final answer
4. **If tool call** -> Execute function, add result to messages, go to step 2
5. **If final answer** -> Return to user

Key insight: LLMs are **stateless**. We must send the FULL conversation history every API call.

In [ ]:
# Initialize OpenAI client
client = OpenAI()  # Reads OPENAI_API_KEY from environment

def run_agent(user_question: str, max_steps: int = 20, verbose: bool = True) -> str:
    """Run the manual agent loop until final answer or max steps reached.

    Args:
        user_question: The user's question to answer
        max_steps: Maximum number of LLM calls to prevent infinite loops (AGENT-08)
        verbose: Print debug information

    Returns:
        The agent's final answer string
    """
    # Per AGENT-04: Initialize conversation with system prompt + user question
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ]

    if verbose:
        print(f"User: {user_question}")
        print("-" * 50)

    # Per AGENT-08: Loop with termination condition
    for step in range(max_steps):
        if verbose:
            print(f"Step {step + 1}/{max_steps}")

        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )

        assistant_message = response.choices[0].message

        # Check if LLM wants to call a tool
        if assistant_message.tool_calls:
            if verbose:
                print(f"  Tool calls: {len(assistant_message.tool_calls)}")

            # Per AGENT-04: Append assistant message to history
            messages.append(assistant_message)

            # Execute each tool call
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  Calling {function_name}({function_args})")

                # Per AGENT-07: Error handling
                try:
                    if function_name == "text_search":
                        result = text_search(**function_args)
                        # Per AGENT-05: JSON-serializable response
                        tool_response = json.dumps({
                            "success": True,
                            "results": result
                        })
                    else:
                        tool_response = json.dumps({
                            "success": False,
                            "error": f"Unknown function: {function_name}"
                        })
                except Exception as e:
                    # Per AGENT-07: Feed error back to LLM
                    tool_response = json.dumps({
                        "success": False,
                        "error": str(e)
                    })
                    if verbose:
                        print(f"  Error: {e}")

                if verbose:
                    # Show first 200 chars of response
                    print(f"  Result: {tool_response[:200]}...")

                # Per AGENT-04, AGENT-05: Append tool result with correct ID
                messages.append({
                    "role": "tool",
                    "content": tool_response,
                    "tool_call_id": tool_call.id
                })
        else:
            # No tool calls = final answer
            if verbose:
                print("  Final answer received")
            return assistant_message.content

    # Per AGENT-08: Max steps reached
    return f"Agent reached max steps ({max_steps}) without completing. Last messages may have context."

print("run_agent() function defined")
print("Ready to answer questions!")

## Testing the Agent

Let's test with some FAQ questions to verify the agent:
1. Invokes text_search tool
2. Uses search results in the answer
3. Handles the full loop correctly

In [ ]:
# Test 1: Simple FAQ question
question1 = "What is the purpose of Docker in the course?"
print("=" * 60)
print("TEST 1: Simple FAQ Question")
print("=" * 60)
answer1 = run_agent(question1)
print(f"\nAnswer: {answer1}")

In [ ]:
# Test 2: More specific question
question2 = "How do I submit homework?"
print("=" * 60)
print("TEST 2: Homework Submission")
print("=" * 60)
answer2 = run_agent(question2)
print(f"\nAnswer: {answer2}")

In [ ]:
# Test 3: Question unlikely to have results (test error handling)
question3 = "What is the airspeed velocity of an unladen swallow?"
print("=" * 60)
print("TEST 3: Question with No Relevant Results")
print("=" * 60)
answer3 = run_agent(question3)
print(f"\nAnswer: {answer3}")

## Understanding the Stateless Pattern

Every API call must include the FULL conversation history. Let's inspect what messages look like after a tool call:

In [ ]:
# Demonstrate stateless pattern by showing message structure
demo_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is Docker?"}
]

# Make one API call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_messages,
    tools=tools
)

assistant_msg = response.choices[0].message

if assistant_msg.tool_calls:
    # Show the structure
    print("After LLM decides to call a tool, messages look like:")
    print()
    print("1. System message (role='system')")
    print("2. User message (role='user')")
    print("3. Assistant message with tool_calls (role='assistant', content=None)")
    print(f"   tool_calls[0].id = '{assistant_msg.tool_calls[0].id}'")
    print(f"   tool_calls[0].function.name = '{assistant_msg.tool_calls[0].function.name}'")
    print(f"   tool_calls[0].function.arguments = '{assistant_msg.tool_calls[0].function.arguments}'")
    print()
    print("4. After executing tool, we add (role='tool', tool_call_id=...)")
    print()
    print("Key insight: tool_call_id MUST match the original tool_calls[].id")
    print("This is how the LLM correlates results with its requests.")

## What Makes This "Agentic"?

The system is **agentic** because:

1. **Autonomous Decision-Making**: The LLM decides WHEN to call tools, not us
2. **Multi-Step Reasoning**: Tool call -> analyze results -> respond (or call again)
3. **Goal-Oriented**: Given a goal (answer question), it figures out HOW to achieve it

This is different from simple prompt -> response because:
- We don't hardcode "search for X"
- The LLM extracts the query from natural language
- The LLM decides if results are sufficient or needs more info

## What This Manual Implementation Shows

Understanding the manual version reveals what frameworks abstract:
1. **Tool Schema**: JSON definition of functions (Pydantic AI auto-generates this)
2. **Message Management**: List of dicts (frameworks handle this internally)
3. **Loop Logic**: while/for with termination (frameworks have built-in limits)
4. **Error Handling**: try/except and feeding errors to LLM (frameworks provide patterns)

Next: Phase 21 will show the same functionality with Pydantic AI - much cleaner code!

## Key Learnings

### Concepts Covered
- **Function Calling**: LLMs can invoke external tools via JSON schema
- **Stateless Pattern**: Every API call needs full conversation history
- **Tool Correlation**: tool_call_id links results to requests
- **Error Handling**: Feed errors to LLM, don't crash

### Implementation Patterns
- **Tool Schema**: `{"type": "function", "function": {...}, "strict": True}`
- **Message Array**: System -> User -> Assistant (tool_calls) -> Tool (result) -> ...
- **Loop Termination**: max_steps prevents infinite loops
- **Graceful Errors**: try/except with JSON error response to LLM

### Trade-offs
| Manual Implementation | Framework (Pydantic AI) |
|----------------------|-------------------------|
| Full control | Abstracted complexity |
| More code (~50 lines) | Less code (~10 lines) |
| Educational value | Production ready |
| Debug easily | Debug through framework |

### Next Steps
- Phase 21: Same functionality with Pydantic AI framework
- Phase 22: Full course notebook with both approaches
- Phase 23: Apply to OWASP corpus